In [1]:
%%html
<style>
  table { margin-left: 0 !important; margin-right: auto !important; }
</style>

In [2]:
from pathlib import Path
import sys

# Move up one level to the project root
root = Path.cwd().parent # Moving the root folder to one parent folder up.
sys.path.append(str(root))


In [3]:
import pandas as pd
import numpy as np

import tensorflow as tf
import networkx as nx
from scipy import sparse

from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

from src.app.ml_models.mdgcn.transductive.model import SupervisedTransductiveModel
from src.app.ml_models.mdgcn.inductive.model import SupervisedInductiveModel


In [4]:
# Collecting dataframes from Elliptic datasets.
features_df = pd.read_csv("../data/elliptic/elliptic_txs_features.csv", header=None)
classes_df = pd.read_csv("../data/elliptic/elliptic_txs_classes.csv")
edges_df = pd.read_csv("../data/elliptic/elliptic_txs_edgelist.csv")

# Collecting important features.
transaction_ids = features_df.iloc[:, 0].values # First column of features dataframe is the transaction ID.
time_steps = features_df.iloc[:, 1].values # Second column of features dataframe is the discrete time step.
node_features = features_df.iloc[:, 2:].values # The rest of 165 columns are features related to bitcoin transactions.

In [5]:
node_features.shape

(203769, 165)

### Build a Crypto-Currency Transaction Graph Data Structure

In [6]:
transactions_graph = nx.DiGraph()

# Every transaction id is a node in my graph.
for transaction in transaction_ids:
    transactions_graph.add_node(transaction)

# Every transaction operation is an edge connection between two nodes in my graph.
for _, row in edges_df.iterrows():
    transactions_graph.add_edge(row["txId1"], row["txId2"])

In [7]:
# transaction_to_index = Dictionary representing every transaction in enumerated order as ID.
transaction_to_index = {
    transaction: index for index, transaction in enumerate(transaction_ids)
}

# N = Length of total transactions.
N = len(transaction_ids)


### Build up Sparse Tensor to store Adjacent List of transactions

Sparse distribution and data collection avoids to store big amount of data (Dense) and improve mathematical calculations in devices with small memory capacity.


In [8]:
def store_features(features, snapshot_dir):

    np.save(f"{snapshot_dir}/node_features.npy", features)
    
def store_adjacent_sparse(adjacent_sparse, hop, N, snapshot_dir):
    
    scipy_sparse = sparse.coo_matrix(
        (
            adjacent_sparse.values.numpy(),
            (
                adjacent_sparse.indices.numpy()[:, 0],
                adjacent_sparse.indices.numpy()[:, 1],
            )
        ),
        shape=(N, N)
    )

    sparse.save_npz(
        f"{snapshot_dir}/adjacent_hop_{hop}.npz",
        scipy_sparse
    )


def compute_adjacent_sparse(graph, transaction_to_index, dist):
    """
        Extract a list of adjacent chain of transactions per each node in the graph.
        The limit of the chain is given by the lengh of the distance defined in `dist`.
    """
    # N = Length of total transactions.
    N = len(transaction_to_index)

    # Collect chain of indices based on distance definition (dist).
    indices_per_hop = [[] for _ in range(dist + 1)]

    # Collect Total adjacent distribution list.
    adjacent_list = []

    # Collect indexes (source and target chain) into indices_per_hop.
    for transaction in graph.nodes():

        # Collect the source index.
        source_index = transaction_to_index[transaction]

        # Find the list of chain transactions from the current transaction (source).
        adjacent_chain = nx.single_source_shortest_path_length(
            graph, transaction, cutoff = dist
        )

        # Collect all chain adjacent index transactions from source transaction,
        # until dist complies.
        for target_transaction, hop in adjacent_chain.items():
            if hop <= dist:
                target_index = transaction_to_index[target_transaction]
                
                indices_per_hop[hop].append([source_index, target_index])


    for hop in range(dist + 1):

        indices = indices_per_hop[hop]

        if len(indices) == 0:
            indices = np.zeros((0, 2), dtype=np.int64)

        values = np.ones(len(indices), dtype=np.float32)

        # Create the SparseTensor directly.
        adjacent_sparse = tf.sparse.SparseTensor(
            indices=indices,
            values=values,
            dense_shape=[N, N]
        )

        # Sort indexes of adjacent_sparse Tensor in canonical row-major order.
        adjacent_sparse = tf.sparse.reorder(adjacent_sparse)

        # Adding sorted Sparse Tensor into the list of adjacent_list variable.
        adjacent_list.append(adjacent_sparse)
        
    return adjacent_list


In [9]:
snapshot_dir = "../data/ml_data"
transaction_dictionary = transaction_to_index
chain_distance = 2

# Build adjacent list of transactions for all the graph.
adjacent_list = compute_adjacent_sparse(
    transactions_graph, 
    transaction_dictionary, 
    chain_distance
)

# Store node_features in disc.
store_features(node_features, snapshot_dir)

# Store adjacent_sparse in disc.
for hop, adjacent_sparse in enumerate(adjacent_list):
    store_adjacent_sparse(adjacent_sparse, hop,len(transaction_dictionary), snapshot_dir)


In [10]:
labels_df = classes_df.set_index("txId")
labels_array = []
labeled_mask = []

for transaction_id in transaction_ids:
    label = labels_df.loc[transaction_id]["class"]

    if label == "unknown":
        labels_array.append(0)      # dummy value
        labeled_mask.append(False)  # not used for loss
    elif label == "1":              # illicit
        labels_array.append(1)
        labeled_mask.append(True)
    else:                           # licit (class "2")
        labels_array.append(0)
        labeled_mask.append(True)

labels = tf.convert_to_tensor(labels_array, dtype=tf.float32)
labels = tf.reshape(labels, (-1, 1))

labeled_mask = np.array(labeled_mask)


### Split Data in Train, Validation and Test (Random Split Mode)

In [11]:
labeled_indices = np.where(labeled_mask)[0]
labeled_labels = labels.numpy()[labeled_indices]

# Train to (temp) split
# This first split will be over the entire data representation,
# the first 80% of split will become training data and the rest 30% will become
# temporal data that later will become into validation and tests.
train_idx, temp_idx = train_test_split(
    labeled_indices,
    stratify=labeled_labels,
    test_size=0.3,
    random_state=42
)

# Temp to (val & test) split
# In this second split the remaining 30% from previous split will be divided
# into 50% Validation data and the other 50% into Test data.
val_idx, test_idx = train_test_split(
    temp_idx,
    stratify=labels.numpy()[temp_idx],
    test_size=0.5,
    random_state=42
)

train_mask = np.zeros(N, dtype=bool)
val_mask   = np.zeros(N, dtype=bool)
test_mask  = np.zeros(N, dtype=bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

train_mask = tf.convert_to_tensor(train_mask)
val_mask = tf.convert_to_tensor(val_mask)
test_mask = tf.convert_to_tensor(test_mask)


In [12]:
print('train_mask.shape', train_mask.shape, np.sum(train_mask))
print('val_mask.shape', val_mask.shape, np.sum(val_mask))
print('test_mask.shape', test_mask.shape, np.sum(test_mask))
print('node_features.shape', node_features.shape)

print(tf.reduce_sum(tf.cast(train_mask, tf.int32)))
print(tf.reduce_sum(tf.cast(val_mask, tf.int32)))

multi = tf.cast(train_mask, tf.int32) * tf.cast(val_mask, tf.int32)
print(tf.reduce_sum(multi))


train_mask.shape (203769,) 32594
val_mask.shape (203769,) 6985
test_mask.shape (203769,) 6985
node_features.shape (203769, 165)
tf.Tensor(32594, shape=(), dtype=int32)
tf.Tensor(6985, shape=(), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)


## Build Transductive Supervised Model

In [13]:
transductive_model = SupervisedTransductiveModel(
    in_dim=node_features.shape[1],
    hidden_dim=64,
    K=2
)

transductive_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    metrics=[
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)


### Transductive Training and Validation Step

In [14]:
transductive_model.fit(
    node_features,
    adjacent_list,
    labels,
    train_mask,
    val_data=(
        node_features,
        adjacent_list,
        labels,
        val_mask
    ),
    epochs=100
)

Epoch 0
Train Loss: 1.081416130065918
Val Loss: 0.37459099292755127
Epoch 10
Train Loss: 0.46057721972465515
Val Loss: 0.2618405818939209
Epoch 20
Train Loss: 0.34810641407966614
Val Loss: 0.23268088698387146
Epoch 30
Train Loss: 0.2964720129966736
Val Loss: 0.19992582499980927
Epoch 40
Train Loss: 0.25991493463516235
Val Loss: 0.17642271518707275
Epoch 50
Train Loss: 0.22872938215732574
Val Loss: 0.1628885269165039
Epoch 60
Train Loss: 0.21713942289352417
Val Loss: 0.15411219000816345
Epoch 70
Train Loss: 0.2057848870754242
Val Loss: 0.14761707186698914
Epoch 80
Train Loss: 0.18300321698188782
Val Loss: 0.14244413375854492
Epoch 90
Train Loss: 0.18381166458129883
Val Loss: 0.13798433542251587


### Transductive Evaluation Step

In [15]:
evaluation_results = transductive_model.evaluate_graph(
    node_features,
    adjacent_list,
    labels,
    test_mask
)

pd.DataFrame([evaluation_results]).T.rename(columns={0: 'Score'})

,Score
loss,0.891566
auc,0.874768
precision,0.891566
recall,0.759531
f1,0.820269


### Split Data in Train, Validation and Test (Temporal Split Mode)

The goal of temporal split is using the field `time_steps` which is the one that separates data by chunks of steps in time with a total of 49 time steps defined in the Elliptic Dataset. This give me the opportunity to test future transactions according the time period.

The reason I am using this field is because it was almost impossible for me to replicate a transaction or simulate a transaction that map the current features (165 features) that comes from the Elliptic Dataset because all of those features are coming with anonimous definition, so there is no way to know what type of value comes in every feature.

This way I am going to train the Inductive model using data that belongs to the time step 1 to 34, then for validation I will use transactions that belong to time steps in the future from 35 to 41 and finally I will only use transactions also in the future from time step 42 to 47, giving the opportunity for simulation purpuses only the last two time steps 48 and 49.

In [16]:
transaction_to_time = {}

for _, row in features_df.iterrows():
    transaction_to_time[row[0]] = row[1]


In [17]:
def getTransactionsByLimit(limit, transactions = transaction_ids, transaction_to_time = transaction_to_time):
    """Return transactions list in a dictionary based on limit"""
    return { transaction for transaction in transactions if transaction_to_time[transaction] <= limit }

def getGraphByType(nodes, edges=edges_df, id_left="txId1", id_right="txId2"):
    """Return graph of temporal data based on partial nodes"""
    
    graph = nx.DiGraph()
    
    for node in nodes:
        graph.add_node(node)
    
    for _, row in edges.iterrows():
    
        source = row[id_left]
        destination = row[id_right]
    
        if source in nodes and destination in nodes:
            graph.add_edge(source, destination)

    return graph


In [18]:
# Calculating new Graph structure based on operation.

graph_train = getGraphByType(getTransactionsByLimit(34))

graph_val = getGraphByType(getTransactionsByLimit(41))

graph_test = getGraphByType(getTransactionsByLimit(47))


In [19]:
# Calculating adjacent list based on operation.

adjacent_train_list = compute_adjacent_sparse(
    graph_train,
    transaction_to_index,
    dist=2
)

adjacent_val_list = compute_adjacent_sparse(
    graph_val,
    transaction_to_index,
    dist=2
)

adjacent_test_list = compute_adjacent_sparse(
    graph_test,
    transaction_to_index,
    dist=2
)

In [20]:
# Temporal Data Split using Masks

train_mask = np.array([
    transaction_to_time[tx] <= 34
    for tx in transaction_ids
])

val_mask = np.array([
    35 <= transaction_to_time[tx] <= 41
    for tx in transaction_ids
])

test_mask = np.array([
    42 <= transaction_to_time[tx] <= 47
    for tx in transaction_ids
])

train_mask = tf.convert_to_tensor(train_mask)
val_mask = tf.convert_to_tensor(val_mask)
test_mask = tf.convert_to_tensor(test_mask)

train_mask = tf.logical_and(train_mask, labeled_mask)
val_mask   = tf.logical_and(val_mask, labeled_mask)
test_mask  = tf.logical_and(test_mask, labeled_mask)


## Build Inductive Supervised Model

In [21]:
inductive_model = SupervisedInductiveModel(
    in_dim=node_features.shape[1],
    hidden_dim=64,
    K=2
)

inductive_model.train_weighted = False
inductive_model.pos_weight = 10.0

inductive_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)


### Inductive Training and Validation Step

In [22]:
inductive_model.fit(
    node_features,
    adjacent_train_list,
    labels,
    train_mask,
    val_data=(
        node_features,
        adjacent_val_list,
        labels,
        val_mask
    ),
    epochs=100
)

Epoch 0
Train Loss: 2.4662280082702637
Val Loss: 2.072025775909424
Epoch 10
Train Loss: 0.7379733920097351
Val Loss: 1.1152722835540771
Epoch 20
Train Loss: 0.40767067670822144
Val Loss: 0.8646736145019531
Epoch 30
Train Loss: 0.3111289441585541
Val Loss: 0.7572094202041626
Epoch 40
Train Loss: 0.25972285866737366
Val Loss: 0.6652796864509583
Epoch 50
Train Loss: 0.22842854261398315
Val Loss: 0.6517110466957092
Epoch 60
Train Loss: 0.20755241811275482
Val Loss: 0.6849216818809509
Epoch 70
Train Loss: 0.18948470056056976
Val Loss: 0.7174849510192871
Epoch 80
Train Loss: 0.17839014530181885
Val Loss: 0.7254485487937927
Epoch 90
Train Loss: 0.16737490892410278
Val Loss: 0.7223286628723145


### Inductive Evaluation Step

In [23]:
inductive_evaluation_results = inductive_model.evaluate_graph(
    node_features,
    adjacent_test_list,
    labels,
    test_mask
)

pd.DataFrame([inductive_evaluation_results]).T.rename(columns={0: 'Score'})


,Score
loss,0.090537
auc,0.703207
precision,0.090537
recall,0.699367
f1,0.160319




# Conclusion after using Multi Distance Models

Reflecting back to the evaluation results from both of the trained models, I can detect that both of them are actually in good shape to generalize important results, even though the two models have different purpuses of executions.

-----------

### Transductive Multi-Distance Graph Convolution Network (MD-GCN):

The objective of this transductive model is to classify nodes (block-chain transactions) within a fixed graph structure by leveraging both features (per node) and the relational structure of the entire transaction network.

It means that the complete graph (Elliptic Dataset) is available during training, including all nodes and their adjacency relationship. Hense the model is able to learn representations using the full graph topology.

The adjacency list of nodes encodes multi-hop transactional relationships, with the goal of detecting anomalous behavior not only for individual transactions but also for structural patterns withing the network.


**Transductive Training and Validation results analysis:**

The idea behind of analysing training an validation results is to detect patterns that makes the model overfit or underfit after certain amount of training repetitions, in this training session I have settup 100 epocs and validation results will be shown every 10th times during the cycle repetition. After the 100 epocs the results were actually good, reflecting no overfitting behavior and also showing smooth decreases of loss.

| Epoc  | Train Loss  | Val  |
|---|---|---|
| 0  | 3.56  | 2.51  |
| 90  | 0.56  | 0.39  |



What these results are showing:
- No clear overfitting
  
- Validation Loss consistenly decreasing

  
- Good convergence stability

  
- Proper learning rate

  
- Stable graph aggregation

  
- No exploiding gradients


**Transductive Evaluation results analysis:**

**AUC (Area Under the ROC Curve) = 0.8559**

We can deduce that:
- The model can separates very well from the fraudulent vs non fraudulent transactions
  
- Shows good ranking ability (Ability to ordering probabilities from most likely to be fraud to least liekely to be fraud)

**Precision = 0.7766 (High)**

This means that the model would preduct fraud with a ~78% of precision.

In terms of finantial point if view this is very good because:

1. False Positives are costly
   
3. Fewer investigations about fraud because of the high precision.

**Recall = 0.5147 (Moderate)**

This means that this model is able to detect arround 51% of all fraudulent transactions.

*- This is an indicative that the model could be in need of an inprovement.*

**F1 (Harmonic mean of precision and recall) = 0.619**

*- This is also a moderate result given the fact that the dataset is inherently unbalanced (this is also a posible candidate to make improvements in the model)*


---------

### Inductive Multi-Distance Graph Convolution Network (MD-GCN):

The goal of Inductive model is to learn a generalizable mapping function that can classify previously unseen nodes (crypto-currency transactions) by leveraging local graph structure and features per transaction.

Using the transaction network structure from Elliptic Dataset, the model is able to aggregate multi-hop neighborhood information to detect anomalous behavior based on both **transactional attributes** and **structural context**.

Inductive model allows to generalize beyond the training graph and perform fraud detection on newly arriving transactions withouth requiring re-train over the full network.

Unlike Transductive model, the inductive model:

- Does not rely on entire fixed graph
  
- Learns to generilize to unseen nodes

  
- Aggregates neighbors locally

  
- Classify new nodes (crypto-currency transactions) withouth re-training on the full graph


**Inductive Training and Validation results analysis:**


| Epoc  | Train Loss  | Val  |
|---|---|---|
| 0  | 49,730  | 22,454  |
| 90  | 882  | 933  |

What these results are showing:

- Train and validation loss decrease steadily.
  
- Curves are very close to each other.

  
- No strong overfitting.

  
- No instability.


**Inductive Evaluation results analysis:**

**AUC (Area Under the ROC Curve) = 0.750**

We can deduce that:
- Only sees local subgraphs.

- Cannot exploit global topology.

- Harder classification problem.

**Precision = 0.614**

Inductive model is less confident when predicting fraud than transductive one.

This suggests:

- It has weaker structural signals.

- It relies more on node-level features.

**Recall = 0.615**

**F1 (Harmonic mean of precision and recall) = 0.614**



### Inductive Model Overall Interpretation

The inductive model achieves a lower AUC (0.75) compared to the transductive counterpart (0.856), indicating reduced discriminative capacity when global graph structure is unavailable.

However, the inductive model demonstrates improved recall (0.615), suggesting better sensitivity to fraudulent transactions at the expense of precision.

This reflects the trade-off between global structural learning (transductive) and generalizable local aggregation (inductive).